# Is it clustered? Calculating Global and local Moran on Ohio counties

This is a paired/small group walkthrough.

Part 1 runs the analysis on median household income. Part 2 is your team's own variable. Part 3 is a stretch goal if there is time.

The data file is `ohio_counties_acs.gpkg`. Its columns are described in `ohio_counties_acs_dictionary.md`, in the same data folder. You will need that page for Part 2. If the course data URL is unavailable, upload the GeoPackage through Colab's Files panel and uncomment the `/content` line in the second code cell.


In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily

## Part 1: median household income, together

### Step 1. Read, inspect, map

Look for: 88 rows and the column list from the data dictionary.

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"
# DATA_BASE_P2 = "/content"   # Colab: after uploading the file through the Files panel

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import collections

ohio = gpd.read_file(f"{DATA_BASE_P2}/ohio_counties_acs.gpkg")
print(ohio.shape, ohio.crs)
print(list(ohio.columns))

fig, ax = plt.subplots(figsize=(7, 7))
ohio.plot(column="median_hh_income", scheme="quantiles", k=5, cmap="YlGnBu",
          legend=True, edgecolor="white", linewidth=0.4, ax=ax,
          legend_kwds={"title": "Median household income (USD)", "loc": "lower left"})
ax.set_title("Median household income, Ohio counties (ACS 2019-2023)")
ax.set_axis_off()
plt.show()

### Step 2. Project, build Queen weights, row-standardize

Look for: `islands: []` and one connected component. If either were different, stop and ask why before going on.

In [ ]:
from libpysal.weights import Queen

ohio_p = ohio.to_crs(26917)

w = Queen.from_dataframe(ohio_p, use_index=False)
print("n:", w.n, "| islands:", w.islands, "| connected components:", w.n_components)

values = list(w.cardinalities.values())
print("neighbors per county: min", min(values), "| max", max(values), "| mean", round(sum(values) / len(values), 2))

w.transform = "r"                           # row-standardize: each row sums to 1

### Step 3. Spatial lag and the value-versus-lag scatter

Look for: the slope of the fitted line. Two numbers print under it. The slope is the one to remember; the correlation is larger because the lag, being an average over neighbors, varies less than the values themselves.

In [ ]:
from libpysal.weights.spatial_lag import lag_spatial

ohio_p["income_lag"] = lag_spatial(w, ohio_p["median_hh_income"].values)

x = ohio_p["median_hh_income"].values
y = ohio_p["income_lag"].values
slope, intercept = np.polyfit(x, y, 1)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x, y, s=25, color="#4C72B0", alpha=0.8)
xs = np.array([x.min(), x.max()])
ax.plot(xs, slope * xs + intercept, color="firebrick")
ax.axvline(x.mean(), color="gray", linestyle="--", linewidth=0.8)
ax.axhline(y.mean(), color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Median household income (USD)")
ax.set_ylabel("Neighbors' average (USD)")
ax.set_title("Income vs. neighbors' average income")
plt.show()

print("slope of the fitted line (lag on value):", round(slope, 4))
print("correlation between income and its spatial lag:", round(np.corrcoef(x, y)[0, 1], 3))

### Step 4. Global Moran's I with permutation inference

Look for: three numbers. 
1. I
2. the expected value under spatial randomness, which is slightly below zero rather than exactly zero
3. `p_sim`, the share of 999 random shuffles that produced an I at least as large as the real one.

In [ ]:
from esda.moran import Moran

np.random.seed(42)                          # set the seed immediately before the permutation test
mi = Moran(ohio_p["median_hh_income"].values, w, permutations=999)

print("Moran's I:", round(mi.I, 4))
print("expected I under spatial randomness:", round(mi.EI, 4), "  (-1 / (n - 1) =", round(-1 / (w.n - 1), 4), ")")
print("z-score (permutation-based):", round(mi.z_sim, 3))
print("p-value (permutation-based, p_sim):", mi.p_sim)

### Step 5. The Moran plot

Look for: the same scatter as step 3, now on standardized axes, with the fitted line whose slope is I.

In [ ]:
from splot.esda import moran_scatterplot

fig, ax = moran_scatterplot(mi)
ax.set_xlabel("Median household income (standardized)")
ax.set_ylabel("Spatial lag of income (standardized)")
ax.set_title("Moran scatterplot, Ohio county median household income")
plt.show()

The four quadrants, reading counter-clockwise from upper right: high values with high neighbors (HH), low values with high neighbors (LH), low values with low neighbors (LL), and high values with low neighbors (HL). HH and LL are clusters. LH and HL are outliers.

### Step 6. Local Moran

Look for: how many of the 88 counties clear the p < 0.05 significance threshold, and how those distributed across the four quadrants. Most counties will not be significant; that is normal.

In [ ]:
from esda.moran import Moran_Local

np.random.seed(42)
lisa = Moran_Local(ohio_p["median_hh_income"].values, w, permutations=999)

significant = lisa.p_sim < 0.05
print(f"{significant.sum()} of {w.n} counties are significant at p < 0.05")

quadrant_labels = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
ohio_p["lisa_quadrant"] = [quadrant_labels[q] for q in lisa.q]
ohio_p["lisa_significant"] = significant
print(collections.Counter(ohio_p.loc[significant, "lisa_quadrant"]))

### Step 7. The LISA cluster map

Look for: where the HH and LL counties sit on the map, and whether any outliers (HL, LH) are present.

In [ ]:
from splot.esda import lisa_cluster

fig, ax = lisa_cluster(lisa, ohio_p, p=0.05, figsize=(8, 8))
ax.set_title("LISA cluster map, median household income\n(Queen contiguity, 999 permutations, p < 0.05)")
plt.show()

### Step 8. Name the clusters

A map of shaded/colored polygons is not meaningful unless you can say what the colored polygons are.

In [ ]:
hh = ohio_p.loc[significant & (ohio_p["lisa_quadrant"] == "HH"), ["name", "median_hh_income"]]
ll = ohio_p.loc[significant & (ohio_p["lisa_quadrant"] == "LL"), ["name", "median_hh_income"]]

print("Significant HH counties (hot spot), highest income first:")
print(hh.sort_values("median_hh_income", ascending=False).to_string(index=False))
print("\nSignificant LL counties (cold spot), lowest income first:")
print(ll.sort_values("median_hh_income").to_string(index=False))

In one sentence each: what region of Ohio is the hot spot, and what region is the cold spot?

_Hot spot:_ 

_Cold spot:_ 

## Part 2: your team's variable

Pick one column from the data dictionary other than `median_hh_income` and `total_population`. Repeat steps 3 through 8 on it. The cells below are seeded with comments and `pass`; scroll up to Part 1 for the syntax.

By the end of the session your notebook should show: 
1. the global I with its p-value
2. the Moran plot
3. the LISA map
4. the named clusters
5. three to five sentences comparing your variable's pattern with income's.

In [ ]:
# Step 3 for your variable: spatial lag and the value-versus-lag scatter.
# Add a new column to ohio_p, as income_lag was added above.
pass

In [ ]:
# Step 4: global Moran's I with 999 permutations.
# Set the seed immediately before the call.
pass

In [ ]:
# Step 5: the Moran plot, with axis labels that name your variable.
pass

In [ ]:
# Step 6: Local Moran, the count significant at p < 0.05, and the quadrant breakdown.
# Use new column names (not lisa_quadrant / lisa_significant) so the income results survive.
pass

In [ ]:
# Step 7: the LISA cluster map, titled with the variable, weights rule, permutations, and threshold.
pass

In [ ]:
# Step 8: the significant HH and LL counties by name.
pass

_Your three to five sentences here: where are the clusters for your variable, and how does that pattern compare with income's?_

## Part 3, if time: a different neighbor rule

Queen contiguity is a choice. Rebuild the weights with k-nearest neighbors, k = 6, and rerun global Moran's I on your variable.

In [ ]:
# Build KNN weights: from libpysal.weights import KNN; KNN.from_dataframe(ohio_p, k=6)
# Row-standardize it, then rerun Moran on your variable with the seed set first.
pass

Did the I statistic change? Does that worry you? How does your choice of neighborhood impact your conclusion?